# Phase 4: Generate environmental features

Derived to help explain heat patterns (SRS Phase 4):
- **NDVI** = (NIR - Red) / (NIR + Red): vegetation intensity (-1 to 1; dense vegetation > 0.5).
- **NDBI** = (SWIR1 - NIR) / (SWIR1 + NIR): built-up intensity (higher = more built-up / bare).
- **NDWI** = (Green - NIR) / (Green + NIR): water-related signal (> 0 usually open water).
- **Land cover:** built-up, vegetation, water and other (from ESA WorldCover 2021).
- Where data quality permits: **elevation**, **distance to major water bodies**, **built-up proportion**, **ERA5-Land temperature and wind**.

**Output:** a feature stack for each city and observation period in `data/processed/features/`. Landsat features are per selected year (March-May median of clear scenes); ERA5-Land is averaged over each MODIS 8-day period. Terminal equivalent: `.venv\Scripts\python.exe scripts\build_features.py`.

**Reading the maps:** axes are UTM zone 43N in kilometres (10 units = 10 km); cyan outline = Mumbai, green = Navi Mumbai; white = no value (outside the cities, or no clear observation).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from urban_heat import acquisition as acq, config, features as ft, logs, plotting as P, preprocessing as pp
from urban_heat.grid import load_grids

log = logs.setup('phase4_notebook')
config.ensure_dirs()
grids = load_grids(config.DATA / 'grids.json')
g1, g30 = grids['modis_1km'], grids['landsat_30m']
if not acq.elevation_path().exists():                       # one-time download (needs internet)
    print(acq.brief(acq.download_elevation(acq.init_ee(), g30)))
ft.run_all()                                               # skips finished files
geoms = pp.city_geoms()
ids1 = pp.read(pp.P_MASKS / 'city_id_1km.tif')['city_id']
ids30 = pp.read(pp.P_MASKS / 'city_id_30m.tif')['city_id']
CHARTS = config.OUTPUTS / 'charts'
YEAR = max(config.LANDSAT_YEARS)
ls = pp.read(config.FEATURES / f'landsat_{YEAR}_30m.tif')
static30 = pp.read(config.FEATURES / 'static_30m.tif')
static1 = pp.read(config.FEATURES / 'static_1km.tif')

def city_stats(a, ids, fmt='{:.2f}', unit=''):
    for k, i in pp.CITY_IDS.items():
        v = a[ids == i]; v = v[np.isfinite(v)]
        print(f"{config.CITIES[k]:12s} mean {fmt.format(v.mean())}{unit}, 10th pct {fmt.format(np.percentile(v, 10))}{unit}, "
              f"90th pct {fmt.format(np.percentile(v, 90))}{unit}  ({v.size} pixels)")

## Figure 4.1: NDVI, vegetation intensity (Landsat, March-May median)
**What it shows:** NDVI at 30 m for the most recent selected year, the per-pixel median of all clear March-May Landsat observations.

**How to read it:** green = vegetation (forest, parks, mangroves, NDVI roughly 0.4-0.8); pale = sparse vegetation or built-up (about 0.1-0.3); brown/red = water or bare surfaces (around or below 0). The printed numbers are city means and 10th/90th percentiles.

In [ ]:
fig, ax = P.raster_map(ls['ndvi'], g30, geoms, cmap='RdYlGn', vmin=-0.2, vmax=0.8, cbar_label='NDVI (unitless, -1 to 1)',
                       title=f'Figure 4.1: NDVI, March-May {YEAR} median (Landsat, 30 m)')
P.save(fig, CHARTS / f'fig4_1_ndvi_{YEAR}.png'); plt.show()
city_stats(ls['ndvi'], ids30)

## Figure 4.2: NDBI, built-up intensity
**What it shows:** NDBI at 30 m, same year and season as Figure 4.1.

**How to read it:** higher (red) = built-up or bare surfaces, which reflect more shortwave infrared than near-infrared; lower (blue) = vegetation and water. Dense urban areas typically sit around 0 to 0.2, vegetation below -0.1.

In [ ]:
fig, ax = P.raster_map(ls['ndbi'], g30, geoms, cmap='RdBu_r', vmin=-0.4, vmax=0.3, cbar_label='NDBI (unitless, -1 to 1)',
                       title=f'Figure 4.2: NDBI, March-May {YEAR} median (Landsat, 30 m)')
P.save(fig, CHARTS / f'fig4_2_ndbi_{YEAR}.png'); plt.show()
city_stats(ls['ndbi'], ids30)

## Figure 4.3: NDWI, water-related signal
**What it shows:** NDWI at 30 m, same year and season.

**How to read it:** values above about 0 (blue) are usually open water: creeks, lakes, the harbour edge. Vegetation is strongly negative, built-up mildly negative.

In [ ]:
fig, ax = P.raster_map(ls['ndwi'], g30, geoms, cmap='BrBG', vmin=-0.6, vmax=0.4, cbar_label='NDWI (unitless, -1 to 1)',
                       title=f'Figure 4.3: NDWI, March-May {YEAR} median (Landsat, 30 m)')
P.save(fig, CHARTS / f'fig4_3_ndwi_{YEAR}.png'); plt.show()
city_stats(ls['ndwi'], ids30)
print('clear observations per pixel (median over city pixels):', int(np.median(ls['clear_obs'][ids30 > 0])))

## Figure 4.4: Land cover in the plan's four groups
**What it shows:** each 30 m pixel's group: built-up (WorldCover built-up), vegetation (tree cover, shrubland, grassland, cropland, herbaceous wetland, mangroves), water (permanent water), other (bare/sparse, snow, moss). The legend gives each group's share of the city area.

**How to read it:** compare with Figure 4.2: built-up areas should coincide with high NDBI.

In [ ]:
grp = static30['landcover_group'].astype(int)
colours = {1: ('Built-up', '#e53935'), 2: ('Vegetation', '#2e7d32'), 3: ('Water', '#1e88e5'), 4: ('Other', '#bdbdbd')}
fig, ax = plt.subplots(figsize=(9, 9))
cmap = ListedColormap([colours[i][1] for i in (1, 2, 3, 4)]); cmap.set_bad('white')
ax.imshow(np.ma.masked_equal(grp, 0), extent=P.extent_km(g30), cmap=cmap, norm=BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], 4),
          interpolation='nearest')
n = (grp > 0).sum()
P.draw_cities(ax, geoms, extra_handles=[Patch(color=c, label=f'{name} ({100 * (grp == i).sum() / n:.1f}% of city area)')
                                        for i, (name, c) in colours.items()])
P.map_axes(ax, f'Figure 4.4: Land-cover groups, ESA WorldCover {config.LANDCOVER_YEAR} (30 m)')
P.save(fig, CHARTS / 'fig4_4_landcover_groups.png'); plt.show()
for k, i in pp.CITY_IDS.items():
    v = grp[ids30 == i]
    print(config.CITIES[k], {colours[g][0]: f'{100 * (v == g).mean():.1f}%' for g in colours})

## Figure 4.5: Built-up proportion per 1 km cell
**What it shows:** for every 1 km model cell, the fraction of its in-city 30 m pixels that are built-up (0 = none, 1 = entirely built-up). This is the "built-up land-cover proportion" input for the models.

**How to read it:** darker = more built-up. It summarises Figure 4.4 at the resolution of the MODIS temperature data.

In [ ]:
fig, ax = P.raster_map(static1['share_built_up'], g1, geoms, cmap='Reds', vmin=0, vmax=1,
                       cbar_label='Share of cell that is built-up (0-1)', title='Figure 4.5: Built-up proportion per 1 km cell')
P.save(fig, CHARTS / 'fig4_5_builtup_share_1km.png'); plt.show()
city_stats(static1['share_built_up'], ids1)

## Figure 4.6: Elevation
**What it shows:** SRTM ground elevation in metres above sea level at 30 m.

**How to read it:** most of both cities lies near sea level; the forested hills of Sanjay Gandhi National Park (north Mumbai) and the ridge east of Navi Mumbai stand out. Elevation affects temperature (cooler higher up) and wind exposure.

In [ ]:
fig, ax = P.raster_map(static30['elevation_m'], g30, geoms, cmap='terrain', vmin=0,
                       vmax=np.nanpercentile(static30['elevation_m'], 99.5), cbar_label='Elevation above sea level (m)',
                       title='Figure 4.6: Elevation, SRTM (30 m)')
P.save(fig, CHARTS / 'fig4_6_elevation.png'); plt.show()
city_stats(static30['elevation_m'], ids30, fmt='{:.0f}', unit=' m')

## Figure 4.7: Distance to the nearest major water body
**What it shows:** straight-line distance from each 30 m pixel to the nearest major water body: a connected area of permanent water of at least 1 km2 (the sea, Thane Creek, the large lakes). Small ponds are ignored.

**How to read it:** darker = closer to water. Water bodies moderate temperature through evaporation and sea breeze, so this tests whether cells far from water are hotter.

In [ ]:
d_km = static30['dist_major_water_m'] / 1000
fig, ax = P.raster_map(d_km, g30, geoms, cmap='viridis_r', vmin=0, cbar_label='Distance to nearest major water body (km)',
                       title='Figure 4.7: Distance to major water bodies (30 m)')
P.save(fig, CHARTS / 'fig4_7_distance_to_water.png'); plt.show()
city_stats(d_km, ids30, fmt='{:.2f}', unit=' km')

## Figure 4.8: ERA5-Land air temperature, 8-day periods, city average
**What it shows:** ERA5-Land 2 m air temperature averaged over each MODIS 8-day period and over each city's cells, 2001-2025. Thin line = every 8-day period; thick line = 1-year rolling mean.

**How to read it:** the regular wave is the seasonal cycle (warmest in April-May, coolest in January). This is regional weather at about 9 km, not neighbourhood temperature; the two cities are close, so their lines nearly overlap.

In [ ]:
rows = []
for y in range(config.START_YEAR, config.END_YEAR + 1):
    e = pp.read(config.FEATURES / f'era5_8day_{y}_1km.tif')
    for name, a in e.items():
        var, d = name.rsplit('_', 1)
        for k, i in pp.CITY_IDS.items():
            rows.append({'date': pd.Timestamp(d), 'var': var, 'city': k, 'value': np.nanmean(a[ids1 == i])})
era = pd.DataFrame(rows)

def era_plot(var, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(13, 4.5))
    for k, colour in [('mumbai', '#0097a7'), ('navi_mumbai', '#43a047')]:
        s = era[(era['var'] == var) & (era.city == k)].set_index('date').value.sort_index()
        ax.plot(s.index, s.values, lw=0.6, alpha=0.5, color=colour, label=f'{config.CITIES[k]}: each 8-day period')
        ax.plot(s.rolling(46, center=True, min_periods=20).mean(), lw=2, color=colour, label=f'{config.CITIES[k]}: 1-year rolling mean')
    ax.set_xlabel('Period start date'); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
    P.save(fig, CHARTS / fname); plt.show()
    s = era[(era['var'] == var)]
    print(s.groupby('city').value.agg(['mean', 'min', 'max']).round(2))

era_plot('air_temp_2m_c', 'Air temperature at 2 m (C)', 'Figure 4.8: ERA5-Land 2 m air temperature, 8-day means, city average',
         'fig4_8_era5_air_temp.png')

## Figure 4.9: ERA5-Land wind speed, 8-day periods, city average
**What it shows:** 10 m wind speed (m/s) averaged over each MODIS 8-day period and each city.

**How to read it:** wind peaks in the monsoon (June-August); calm periods favour surface heating. 1 m/s = 3.6 km/h.

In [ ]:
era_plot('wind_speed_10m_ms', 'Wind speed at 10 m (m/s)', 'Figure 4.9: ERA5-Land 10 m wind speed, 8-day means, city average',
         'fig4_9_era5_wind.png')

## Figure 4.10: How NDVI and NDBI change across the selected years
**What it shows:** the city-average NDVI (solid) and NDBI (dashed) for each selected Landsat year (March-May medians).

**How to read it:** falling NDVI with rising NDBI would indicate loss of vegetation to built-up land. Year-to-year differences also include differences in that season's cloud-free observations, so small changes should not be over-interpreted.

In [ ]:
summ = pd.read_csv(config.FEATURES / 'feature_summary.csv')
yr = summ[summ.period.str.endswith('Mar-May') & summ.feature.isin(['ndvi', 'ndbi'])].copy()
yr['year'] = yr.period.str[:4].astype(int)
fig, ax = plt.subplots(figsize=(10, 4.5))
for k, colour in [('mumbai', '#0097a7'), ('navi_mumbai', '#43a047')]:
    for feat, ls_ in [('ndvi', '-'), ('ndbi', '--')]:
        d = yr[(yr.city == k) & (yr.feature == feat)].sort_values('year')
        ax.plot(d.year, d['mean'], ls_, marker='o', color=colour, label=f'{config.CITIES[k]} {feat.upper()}')
ax.set_xticks(sorted(yr.year.unique())); ax.set_xlabel('Year (March-May)'); ax.set_ylabel('City-average index value')
ax.set_title('Figure 4.10: City-average NDVI and NDBI by selected year'); ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
P.save(fig, CHARTS / 'fig4_10_indices_by_year.png'); plt.show()
print(yr.pivot_table(index='year', columns=['city', 'feature'], values='mean').round(3))

## The feature stack
Every file below covers both cities on the common grids; a city's stack is its cells in these files (city-id masks in `data/processed/masks/`). `feature_summary.csv` lists each feature's mean, min, max and valid share per city and period.

In [ ]:
for p in sorted(config.FEATURES.glob('*')):
    print(f'{p.name:32s} {p.stat().st_size / 1e6:7.1f} MB')